# Lab 8 · Tính toán và so sánh theo thời gian

**Lập trình xử lý dữ liệu · 2627-1 · Giờ thực hành · bài 8**

Chọn **Save a copy in Drive** trước khi sửa.

Lab gồm năm phần: kiểm cách đọc ngày, tổng hợp theo quý, cửa sổ trượt bảy ngày,
so cùng kỳ và một bài tự làm.
Bảng đặt phòng đầu bài là **giả lập**. Dữ liệu thật là đánh giá tại Santiago,
snapshot **29/06/2026**; mỗi hàng là một đánh giá, `date` là ngày ghi nhận.

## Cách làm việc trong bài lab

- Điền các ô `TODO`; đọc kết quả và giải thích bằng dữ liệu gốc.
  `assert` kiểm một số điều kiện, không thay thế việc xem các trường hợp sai.
- Với các phần có hướng dẫn, tự viết trước khi nhờ AI.
  Các bài kiểm tra trên lớp (🚫 đóng) đánh giá các kỹ năng này.
- Bài tự làm ✅ mở cho phép dùng AI, kèm khai báo và cách kiểm chứng.
- Khi cần hỗ trợ, trao đổi với giảng viên thực hành về yêu cầu hoặc trường hợp chưa hiểu.

## Mục tiêu

1. Đọc ngày theo đúng định dạng của nguồn; phân biệt ngày không tồn tại với ngày bị thiếu.
2. Chọn đúng lịch và phép tổng hợp; nhận diện kỳ chưa trọn.
3. Giải thích các giá trị đầu chuỗi và khoảng thời gian của cửa sổ trượt.
4. So sánh các giai đoạn tương ứng và giữ rõ kỳ gốc.
5. Tính theo số đánh giá mỗi ngày khi so các tháng có số ngày khác nhau.

## Chuẩn bị

Chạy hai ô dưới trước. Phiên bản pandas được ghim để khớp với ví dụ trên slide.
Nếu phiên Colab đã nạp pandas phiên bản khác, chọn **Restart session and run all** sau khi cài.

In [ ]:
%pip -q install pandas==3.0.3

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

print("pandas:", pd.__version__)

## Phần 1 · Kiểm cách đọc ngày

### Tính số đêm lưu trú

Mỗi hàng là một lượt đặt. Nguồn quy định **DD/MM/YYYY**; có một ngày không
tồn tại và một ngày bị thiếu. Tạo hai Series `vao`, `ra` có kiểu ngày giờ, rồi
tính Series `so_dem` (số đêm của từng lượt đặt). Nếu thiếu một trong hai ngày hợp lệ,
để `so_dem` là giá trị thiếu.

`errors="coerce"` chuyển ngày không đọc được thành `NaT`.
`.dt.days` lấy số ngày của từng khoảng thời gian.

In [ ]:
dat = pd.DataFrame({
    "vao": ["28/02/2026", "03/07/2026", "31/06/2026", None],
    "ra": ["03/03/2026", "06/07/2026", "05/07/2026", "12/07/2026"],
}, index=["D1", "D2", "D3", "D4"])
print(dat)

In [ ]:
# TODO: đổi kiểu theo quy ước nguồn và tính số đêm.
vao = None
ra = None
so_dem = None

assert so_dem.loc[["D1", "D2"]].tolist() == [3, 3]
assert so_dem.loc[["D3", "D4"]].isna().all()
print(dat.assign(so_dem=so_dem))

### Xác định ngày không đọc được

D3 và D4 cùng cho kết quả thiếu nhưng nguyên nhân khác nhau.
Tạo mask `loi_vao` chỉ đánh dấu ngày vào **có trong nguồn nhưng không đọc được**.
Cần đối chiếu với giá trị gốc trong `dat`, vì riêng `vao` không phân biệt được hai trường hợp.

In [ ]:
# TODO: chỉ đánh dấu lỗi chuyển đổi, không đánh dấu ngày đã thiếu từ nguồn.
loi_vao = None

assert loi_vao[loi_vao].index.tolist() == ["D3"]
print(dat.loc[loi_vao])

## Phần 2 · Tổng hợp đánh giá theo quý

### Đọc đúng snapshot

Dùng `visualisations/reviews.csv` của Santiago, mốc chụp **29/06/2026**.
File cache cục bộ có cùng dữ liệu; Colab tải đúng URL nếu chưa có cache.

In [ ]:
URL_REVIEWS = ("https://data.insideairbnb.com/chile/rm/santiago/"
               "2026-06-29/visualisations/reviews.csv")
tep = Path("data/reviews_santiago_full_2026-06-29.csv.gz")
nguon = tep if tep.exists() else URL_REVIEWS
rv_raw = pd.read_csv(nguon, usecols=["listing_id", "date"])
print(rv_raw.head(3))
assert len(rv_raw) == 690112

### Giới hạn phạm vi ngày

Đổi cột `date` theo `YYYY-MM-DD`. Tạo bảng con `ngoai_moc` (một DataFrame giữ
cả hai cột `listing_id` và `date`) gồm các hàng ghi ngày sau mốc chụp; giữ phần
còn lại trong `rv`. Đặt ngày làm chỉ mục của `r` và sắp theo thời gian.
Không xoá các đánh giá chỉ vì trùng ngày.

In [ ]:
moc = pd.Timestamp("2026-06-29")

# TODO: đổi kiểu, kiểm phạm vi, lọc và đặt chỉ mục.
ngoai_moc = None
rv = None
r = None

assert isinstance(ngoai_moc, pd.DataFrame)
assert len(ngoai_moc) == 204
assert len(r) == 689908
assert isinstance(r.index, pd.DatetimeIndex)
assert r.index.is_monotonic_increasing
assert r.index.max() <= moc
print(ngoai_moc["date"].value_counts().sort_index())

### Chọn các quý đã kết thúc

Tạo `theo_quy` đếm đánh giá theo quý với `resample("QE").size()`.
`QE` gắn nhãn cuối quý: 31/3, 30/6, 30/9, 31/12.
Giữ các quý đã kết thúc tại mốc chụp trong `quy_day_du`.
Dữ liệu chỉ có ngày, nên có thể đối chiếu trực tiếp nhãn cuối quý với `moc`.

In [ ]:
# TODO: đếm và chọn quý đã kết thúc.
theo_quy = None
quy_day_du = None

assert theo_quy.sum() == len(r)
assert theo_quy.loc["2026-06-30"] == 60064
assert quy_day_du.index[-1] == pd.Timestamp("2026-03-31")
assert quy_day_du.loc["2026-03-31"] == 68395
print(theo_quy.tail(3))

Quý II/2026 được gắn nhãn 30/6 dù dữ liệu chỉ đến 29/6.
Một báo cáo dùng hai con số cuối bảng để kết luận:
**“Số đánh giá quý II/2026 giảm so với quý I.”**

Đếm số ngày đã quan sát của mỗi quý, rồi viết một câu nhận xét về kết luận này.

**Nhận xét của bạn:** …

### Kiểm cách chọn kỳ cuối

Chuỗi giả lập dưới có số đánh giá của hai quý. Chọn các quý đã kết thúc tại mốc chụp
giả định 29/06/2026 (lưu vào `truoc_ket_thuc`) và 30/06/2026 (lưu vào `ngay_ket_thuc`).
Cách lọc đúng cho kết quả khác nhau ở hai mốc chụp: không phải lúc nào cũng bỏ quý cuối.

In [ ]:
dem_quy = pd.Series([100, 120], index=pd.to_datetime([
    "2026-03-31", "2026-06-30",
]))

In [ ]:
# TODO: chọn các quý đã kết thúc cho từng mốc chụp.
truoc_ket_thuc = None
ngay_ket_thuc = None

assert len(truoc_ket_thuc) == 1
assert len(ngay_ket_thuc) == 2

## Phần 3 · Cửa sổ trượt bảy ngày

### Tìm giai đoạn có nhiều đánh giá

Với mỗi ngày trong năm 2025, tính số đánh giá trung bình của 7 ngày kết thúc
tại ngày đó: chính ngày đó và sáu ngày trước. **Ngày nào có trung bình này lớn nhất?**
Chỉ dùng các ngày trong năm 2025; không dùng dữ liệu sau ngày đang xét.

Tạo chuỗi `theo_ngay` cho cả năm với `resample("D").size()`, rồi `tb7`.
Viết rõ `min_periods=7`; dùng `idxmax()` để lấy nhãn ngày có giá trị lớn nhất.

**Dự đoán trước khi chạy:** sáu giá trị đầu của `tb7` (01/01–06/01/2025) là gì? Vì sao? …

In [ ]:
# TODO: tính theo lịch ngày, rồi tìm cửa sổ có trung bình lớn nhất.
theo_ngay = None
tb7 = None
ngay_cao_nhat = None

assert len(theo_ngay) == 365
assert theo_ngay.sum() == 211432
assert tb7.iloc[:6].isna().all()
assert ngay_cao_nhat == pd.Timestamp("2025-11-26")
print(tb7.loc[ngay_cao_nhat])

Đối chiếu bằng bảy giá trị gốc trong cửa sổ kết thúc ở ngày vừa tìm.
`pd.Timedelta(days=6)` là khoảng sáu ngày; trừ nó khỏi `ngay_cao_nhat` để lấy
ngày đầu cửa sổ.

In [ ]:
dau = ngay_cao_nhat - pd.Timedelta(days=6)
cua_so = theo_ngay.loc[dau:ngay_cao_nhat]
assert len(cua_so) == 7
assert abs(cua_so.mean() - tb7.loc[ngay_cao_nhat]) < 1e-9
print(cua_so)

### Xem khoảng thời gian vừa tính

Chạy ô vẽ sẵn dưới để xem số đánh giá mỗi ngày và đường trung bình 7 ngày
trong tháng 11/2025. Không suy ra nguyên nhân từ hình này.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
theo_ngay.loc["2025-11"].plot(ax=ax, color="0.65", label="Mỗi ngày")
tb7.loc["2025-11"].plot(ax=ax, label="Trung bình 7 ngày gần nhất")
ax.set_xlabel("Ngày")
ax.set_ylabel("Số đánh giá")
ax.legend()
plt.show()

**Giải thích của bạn:** Bản thân ngày 26/11 có bao nhiêu đánh giá? Những ngày nào
trong cửa sổ kéo trung bình lên? Vì sao đỉnh của đường trung bình nằm sau đỉnh
của đường số đánh giá mỗi ngày? …

## Phần 4 · So cùng kỳ

### So quý I qua hai năm

Một đoạn code AI sinh dùng chuỗi `quy_day_du`:

```python
cung_ky = quy_day_du / quy_day_du.shift(12) - 1
```

**Sửa phép tính để đo mức thay đổi số đánh giá quý I/2026 so với quý I/2025.**
`shift` đếm hàng; ở đây một hàng là một quý. Không cần đổi bảng sang tháng.

In [ ]:
# TODO: dùng đúng kỳ gốc.
cung_ky = None

muc_doi = cung_ky.loc["2026-03-31"]
quy_goc = quy_day_du.loc["2025-03-31"]
quy_nay = quy_day_du.loc["2026-03-31"]
assert abs(muc_doi - (quy_nay / quy_goc - 1)) < 1e-9
print(f"{quy_goc:,} → {quy_nay:,}: {muc_doi:.1%}")

### Đối chiếu với ngày gốc

Tính số đánh giá quý I/2025 bằng cách lọc ngày trực tiếp, độc lập với `resample`.
Khoảng cần lấy là từ **01/01/2025 đến trước 01/04/2025**.

In [ ]:
# TODO: đếm trực tiếp trong rv bằng mask ngày.
dem_truc_tiep = None

assert dem_truc_tiep == quy_day_du.loc["2025-03-31"]

**Giải thích của bạn:** `shift(12)` trong chuỗi quý đã so với thời điểm nào? …

## Phần 5 · Bài tự làm ✅ mở (làm sớm tại lớp hoặc làm tại nhà)

### So tháng 2 với tháng 1 theo số đánh giá mỗi ngày

Hình «Phân bố đánh giá theo tháng» ở Bài 8 dựa trên tổng số đánh giá của từng
tháng qua bốn năm 2022–2025, quy về chỉ số với 100 là mức trung bình của 12 tháng.
Số đánh giá tăng khoảng 1,6 lần mỗi năm nên các tháng cuối năm cao hơn một phần
chỉ vì xu hướng. Hình chưa tách xu hướng và chưa điều chỉnh số ngày từng tháng.
Phần này chỉ xử lý số ngày: tháng 2 ngắn hơn tháng 1 ba ngày (hai ngày nếu là năm nhuận).

**Tháng 2 thay đổi bao nhiêu phần trăm so với tháng 1, theo tổng số đánh giá và
theo số đánh giá mỗi ngày?**

Lấy tháng 1 làm kỳ gốc và tính mức thay đổi như `muc_doi` ở Phần 4:
tháng 2 / tháng 1 − 1, viết dạng tỷ lệ (`-0.05` nghĩa là giảm 5%, không nhân 100).
Số đánh giá mỗi ngày của một tháng là tổng số đánh giá của tháng đó trong bốn năm
chia cho tổng số ngày của tháng đó trong bốn năm.
`.index.days_in_month` lấy số ngày của từng tháng, kể cả tháng 2 của năm nhuận.
Ô dưới chuẩn bị số đếm theo tháng và số ngày tương ứng.

In [ ]:
thang = r.resample("ME").size().loc["2022":"2025"]
so_ngay = pd.Series(thang.index.days_in_month, index=thang.index)
assert len(thang) == 48
assert so_ngay.loc["2024-02-29"] == 29

In [ ]:
# TODO: gộp cùng tháng qua bốn năm, tính số đánh giá mỗi ngày,
# rồi so tháng 2 với tháng 1 theo hai cách.
# chenh_tong, chenh_ngay: tháng 2 / tháng 1 - 1, dạng tỷ lệ (-0.05 là giảm 5%).
tong = None
tb_ngay = None
chenh_tong = None
chenh_ngay = None

assert tong.index.tolist() == list(range(1, 13)), "Chỉ mục phải là số tháng 1–12."
assert tong.sum() == thang.sum(), "tong: cộng số đánh giá của cùng tháng qua bốn năm."
assert tb_ngay.index.tolist() == list(range(1, 13)), "Chỉ mục phải là số tháng 1–12."
cach_tinh = ("Mỗi tháng: tổng số đánh giá trong bốn năm chia cho "
             "tổng số ngày của tháng đó trong bốn năm.")
assert round(tb_ngay.loc[1], 2) == 253.23, cach_tinh
assert round(tb_ngay.loc[2], 2) == 230.52, cach_tinh
assert round(tb_ngay.loc[12], 2) == 352.45, cach_tinh
dang_ty_le = ("Mức thay đổi = tháng 2 / tháng 1 - 1, dạng tỷ lệ: "
              "số âm là giảm, không nhân 100.")
assert abs(chenh_tong - (tong.loc[2] / tong.loc[1] - 1)) < 1e-9, (
    f"chenh_tong = {chenh_tong}. {dang_ty_le}")
assert abs(chenh_ngay - (tb_ngay.loc[2] / tb_ngay.loc[1] - 1)) < 1e-9, (
    f"chenh_ngay = {chenh_ngay}. {dang_ty_le}")
print(f"Tháng 2 so với tháng 1: theo tổng {chenh_tong:.1%}, "
      f"theo số đánh giá mỗi ngày {chenh_ngay:.1%}")

**Giải thích của bạn:** Tổng số ngày của tháng 2 trong bốn năm ít hơn tháng 1 bao nhiêu
phần trăm? So với `chenh_tong`, phần do số ngày chiếm khoảng bao nhiêu? Sau khi tính
theo số đánh giá mỗi ngày, tháng 2 có còn thấp hơn tháng 1 không? …

**Khai báo AI (nếu dùng):** công cụ, yêu cầu chính, một kết quả bạn đã tự kiểm. …

## Tóm tắt bài lab

- `coerce` cho phép tiếp tục, nhưng phải phân biệt lỗi chuyển đổi với giá trị thiếu từ nguồn.
- Nhãn cuối kỳ không cho biết kỳ đó đã trọn hay chưa.
- Trung bình trượt gắn ở ngày cuối cửa sổ: các giá trị đầu thiếu khi chưa đủ bảy
  ngày, và đỉnh của đường trung bình có thể nằm sau ngày có số đánh giá cao nhất.
- So cùng kỳ phải theo đúng tần suất của chuỗi và cùng phạm vi quan sát.
- Tổng theo tháng phụ thuộc số ngày của tháng; khi so các tháng, xem cả số đánh giá mỗi ngày.

Chạy **Restart session and run all**, rồi kiểm tra: mọi ô code đều có kết quả; các ô
nhận xét, giải thích và khai báo AI (nếu dùng) đã điền đủ.

**Đọc thêm:** [McKinney chương 11](https://wesmckinney.com/book/time-series),
[pandas: dữ liệu thời gian](https://pandas.pydata.org/docs/user_guide/timeseries.html).

**Bài sau:** thi giữa kỳ; sau đó học làm sạch dữ liệu có cấu trúc ở Bài 10.